In [4]:
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import matplotlib.pyplot as plt
import joblib

In [5]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**MLP dùng thư viện**

In [6]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# If df not loaded:
df = pd.read_csv("/content/data_for_training_final.csv")

# Repro
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Prepare data
X = df.drop("log_price", axis=1).to_numpy(dtype=np.float32)
y = df["log_price"].to_numpy(dtype=np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale (CPU)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train/val split
val_frac = 0.1
perm = np.random.permutation(len(X_train_scaled))
val_size = max(1, int(len(X_train_scaled) * val_frac))
val_idx = perm[:val_size]
train_idx = perm[val_size:]

X_tr = X_train_scaled[train_idx]
y_tr = y_train[train_idx]
X_val = X_train_scaled[val_idx]
y_val = y_train[val_idx]

# Torch tensors
X_tr_t = torch.tensor(X_tr, dtype=torch.float32, device=device)
y_tr_t = torch.tensor(y_tr, dtype=torch.float32, device=device).view(-1, 1)
X_val_t = torch.tensor(X_val, dtype=torch.float32, device=device)
y_val_t = torch.tensor(y_val, dtype=torch.float32, device=device).view(-1, 1)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32, device=device)

train_loader = DataLoader(
    TensorDataset(X_tr_t, y_tr_t), batch_size=64, shuffle=True
)

# MLP model (tanh, (100,100,50))
class MLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 100),
            nn.Tanh(),
            nn.Linear(100, 100),
            nn.Tanh(),
            nn.Linear(100, 50),
            nn.Tanh(),
            nn.Linear(50, 1),
        )

    def forward(self, x):
        return self.net(x)

model = MLP(X_tr.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.001, weight_decay=0.0001
)

# Training with early stopping
max_epochs = 2000
patience = 50
best_val = float("inf")
best_state = None
wait = 0

for epoch in range(1, max_epochs + 1):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t)
        val_loss = criterion(val_pred, y_val_t).item()

    if val_loss < best_val - 1e-6:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            break

if best_state is not None:
    model.load_state_dict(best_state)

# Evaluation
model.eval()
with torch.no_grad():
    y_pred_log = model(X_test_t).cpu().numpy().ravel()

mae_log = mean_absolute_error(y_test, y_pred_log)
r2_log = r2_score(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

y_test_real = np.exp(y_test)
y_pred_real = np.exp(y_pred_log)
mae_real = mean_absolute_error(y_test_real, y_pred_real)

print(f"R2: {r2_log:.4f}")
print(f"MAE (log): {mae_log:.4f}")
print(f"RMSE (log): {rmse_log:.4f}")
print(f"MAE (real): {mae_real:,.0f} VND")

Device: cuda
R2: 0.7375
MAE (log): 0.2766
RMSE (log): 0.3631
MAE (real): 3,306,445,824 VND


**MLP không dùng thư viện**

In [ ]:
import numpy as np

np.random.seed(42)

def tanh(x): return np.tanh(x)
def tanh_grad(x): return 1.0 - np.tanh(x) ** 2

def r2_score_np(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - ss_res / ss_tot

def glorot_uniform(fan_in, fan_out):
    limit = np.sqrt(6.0 / (fan_in + fan_out))
    return np.random.uniform(-limit, limit, size=(fan_in, fan_out))

def init_params(layer_sizes):
    params = []
    for fan_in, fan_out in zip(layer_sizes[:-1], layer_sizes[1:]):
        W = glorot_uniform(fan_in, fan_out)
        b = np.zeros((1, fan_out))
        params.append({"W": W, "b": b})
    return params

def forward(X, params):
    a = X
    caches = []
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        z = a @ W + b
        a_prev = a
        if i < len(params) - 1:
            a = tanh(z)
        else:
            a = z  # linear output
        caches.append((a_prev, z))
    return a, caches

def backward(X, y, y_pred, params, caches, alpha):
    m = X.shape[0]
    dA = (2.0 / m) * (y_pred - y)
    grads = []

    for i in reversed(range(len(params))):
        a_prev, z = caches[i]
        W = params[i]["W"]

        if i == len(params) - 1:
            dZ = dA
        else:
            dZ = dA * tanh_grad(z)

        dW = a_prev.T @ dZ + alpha * W
        db = np.sum(dZ, axis=0, keepdims=True)
        dA = dZ @ W.T

        grads.insert(0, {"dW": dW, "db": db})
    return grads

def sgd_nesterov_update(params, grads, velocity, lr, momentum):
    for i in range(len(params)):
        vW_prev = velocity[i]["W"]
        vB_prev = velocity[i]["b"]

        vW = momentum * vW_prev - lr * grads[i]["dW"]
        vB = momentum * vB_prev - lr * grads[i]["db"]

        params[i]["W"] += -momentum * vW_prev + (1 + momentum) * vW
        params[i]["b"] += -momentum * vB_prev + (1 + momentum) * vB

        velocity[i]["W"], velocity[i]["b"] = vW, vB

X_train = X_train_scaled
X_test = X_test_scaled

val_frac = 0.1
perm = np.random.permutation(len(X_train))
val_size = max(1, int(len(X_train) * val_frac))
val_idx = perm[:val_size]
train_idx = perm[val_size:]

X_tr, y_tr = X_train[train_idx], y_train[train_idx].reshape(-1, 1)
X_val, y_val = X_train[val_idx], y_train[val_idx].reshape(-1, 1)

layer_sizes = [X_tr.shape[1], 100, 100, 50, 1]
params = init_params(layer_sizes)

velocity = [{"W": np.zeros_like(p["W"]), "b": np.zeros_like(p["b"])} for p in params]

lr = 0.001
alpha = 0.0001
momentum = 0.9
max_iter = 4000
tol = 1e-4
n_iter_no_change = 10

batch_size = min(200, len(X_tr))
best_val = -np.inf
best_params = None
wait = 0

for epoch in range(1, max_iter + 1):
    # shuffle
    perm2 = np.random.permutation(len(X_tr))
    X_tr, y_tr = X_tr[perm2], y_tr[perm2]

    # mini-batch
    for start in range(0, len(X_tr), batch_size):
        end = start + batch_size
        xb = X_tr[start:end]
        yb = y_tr[start:end]

        y_pred, caches = forward(xb, params)
        grads = backward(xb, yb, y_pred, params, caches, alpha)
        sgd_nesterov_update(params, grads, velocity, lr, momentum)

    # early stopping on val R2
    y_val_pred, _ = forward(X_val, params)
    val_r2 = r2_score_np(y_val.ravel(), y_val_pred.ravel())

    if val_r2 > best_val + tol:
        best_val = val_r2
        best_params = [ {"W": p["W"].copy(), "b": p["b"].copy()} for p in params ]
        wait = 0
    else:
        wait += 1
        if wait >= n_iter_no_change:
            break

if best_params is not None:
    params = best_params

y_pred_log = forward(X_test, params)[0].ravel()

mae_log = np.mean(np.abs(y_test - y_pred_log))
rmse_log = np.sqrt(np.mean((y_test - y_pred_log) ** 2))
r2_log = r2_score_np(y_test, y_pred_log)

y_test_real = np.exp(y_test)
y_pred_real = np.exp(y_pred_log)
mae_real = np.mean(np.abs(y_test_real - y_pred_real))

print("\nManual MLP:")
print(f"R2: {r2_log:.4f}")
print(f"MAE (log): {mae_log:.4f}")
print(f"RMSE (log): {rmse_log:.4f}")
print(f"MAE (real): {mae_real:,.0f} VND")


Manual MLP:
R2: 0.7492
MAE (log): 0.2716
RMSE (log): 0.3549
MAE (real): 3,187,430,201 VND
